# AI Interview Coach - Preference Scoring (RLAIF)

This notebook runs the judge model to compare pairs of feedback responses and determine which one is better. This is used to build the preference dataset for RLAIF/DPO training.

## 1. Mount Google Drive
Mounting Drive ensures that your results (`.jsonl` files) are saved permanently even if the Colab runtime is disconnected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to the location of your project in Google Drive
PROJECT_PATH = "/content/drive/MyDrive/ai-interview-coach"

import os
if not os.path.exists(PROJECT_PATH):
    print(f"Creating project directory: {PROJECT_PATH}")
    !git clone https://github.com/dcyforjob2020/ai-interview-coach.git "{PROJECT_PATH}"

%cd "{PROJECT_PATH}"
# Pull latest changes
!git pull origin main

## 2. Setup Environment
Install the necessary libraries for running large language models and quantization.

In [ ]:
# Install project dependencies
!pip install -U transformers accelerate bitsandbytes torch
!pip install -r requirements.txt

print("\nSetup complete. Please restart the runtime if you hit any OOM or library errors.")

## 3. Configuration
Adjust the paths and model as needed. 
- `INPUT_PATH`: The file containing candidates to be compared. (e.g., `train/preference_candidates.jsonl` for all data)
- `OUTPUT_PATH`: Where the resulting preference pairs will be saved.
- `MODEL_NAME`: The judge model to use (default: `Qwen/Qwen3.5-9B`).
- `QUANTIZE`: Enable 4-bit quantization to fit the model on consumer GPUs.

In [ ]:
# List available candidate files
!ls train/*.jsonl

INPUT_PATH = "train/preference_candidates.jsonl"  # Set to the file containing your data
OUTPUT_PATH = "train/preference_pairs.jsonl"       # Where to save the results
MODEL_NAME = "Qwen/Qwen3.5-9B"
QUANTIZE = True
LIMIT = None  
OVERWRITE = False

## 4. Run Preference Scoring (Whole Dataset)
Run this cell to process the entire input file in one go.

In [ ]:
limit_arg = f"--limit {LIMIT}" if LIMIT else ""
quantize_arg = "--quantize" if QUANTIZE else ""
overwrite_arg = "--overwrite" if OVERWRITE else ""

!python eval/score_preferences.py \
    --input {INPUT_PATH} \
    --output {OUTPUT_PATH} \
    --model {MODEL_NAME} \
    {limit_arg} \
    {quantize_arg} \
    {overwrite_arg}

## 5. Split Input & Run in Parts (Optional)
If the dataset is too large, use these cells to split it and run in two parts.

In [ ]:
import json
from pathlib import Path

def split_jsonl(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    total = len(lines)
    mid = total // 2
    
    part1_path = Path(input_file).with_name(Path(input_file).stem + "_part1.jsonl")
    part2_path = Path(input_file).with_name(Path(input_file).stem + "_part2.jsonl")
    
    with open(part1_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines[:mid]) + "\n")
        
    with open(part2_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines[mid:]) + "\n")
        
    print(f"Split {total} records into:")
    print(f" - {part1_path} ({mid} records)")
    print(f" - {part2_path} ({total - mid} records)")
    return part1_path, part2_path

if os.path.exists(INPUT_PATH):
    PART1_INPUT, PART2_INPUT = split_jsonl(INPUT_PATH)
else:
    print(f"Error: INPUT_PATH not found: {INPUT_PATH}")

In [ ]:
if 'PART1_INPUT' not in globals():
    print("Error: Run the split cell above first.")
else:
    limit_arg = f"--limit {LIMIT}" if LIMIT else ""
    quantize_arg = "--quantize" if QUANTIZE else ""
    overwrite_arg = "--overwrite" if OVERWRITE else ""

    !python eval/score_preferences.py \
        --input {PART1_INPUT} \
        --output {OUTPUT_PATH} \
        --model {MODEL_NAME} \
        {limit_arg} \
        {quantize_arg} \
        {overwrite_arg}

In [ ]:
if 'PART2_INPUT' not in globals():
    print("Error: Run the split cell above first.")
else:
    limit_arg = f"--limit {LIMIT}" if LIMIT else ""
    quantize_arg = "--quantize" if QUANTIZE else ""
    overwrite_arg = "--overwrite" if OVERWRITE else ""

    !python eval/score_preferences.py \
        --input {PART2_INPUT} \
        --output {OUTPUT_PATH} \
        --model {MODEL_NAME} \
        {limit_arg} \
        {quantize_arg} \
        {overwrite_arg}

## 6. Inspect Results
Load the results and look at the win rate between Feedback A and Feedback B.

In [ ]:
import json
import pandas as pd
from pathlib import Path

if Path(OUTPUT_PATH).exists():
    records = []
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    
    if records:
        df = pd.DataFrame(records)
        print(f"Total records scored: {len(df)}")
        
        if 'winner' in df.columns:
            print("\nWinner distribution:")
            print(df['winner'].value_counts())
            
        print("\nSample record:")
        display(df.head(1))
    else:
        print("Output file is empty.")
else:
    print(f"Output file not found: {OUTPUT_PATH}. Check if the scoring run completed successfully.")